# T5 · Cross-filter direction

The default relationship direction is `Single`: filters flow from dimensions to the fact table. `Both` is not used for these six situations because it can make dimensions filter one another through the fact, creating unintended filter paths, ambiguity, or misleading totals.

| # | Situation | Verdict | Why — and what breaks if you get it wrong |
|---:|---|---|---|
| 1 | A slicer on `DimMode` must filter the shipment totals on every visual. | **Single** | The mode dimension only needs to filter `FactShipment`. `Both` is unnecessary; using it can let fact filters flow back into dimensions and create unintended interactions or ambiguous paths. |
| 2 | `DimCity` and `DimMode` are both set to `Both`. Power BI refuses the second relationship. | **Single** | Keep both dimension-to-fact relationships single-direction. Making both bidirectional creates competing paths through `FactShipment`, so Power BI can detect ambiguity and refuse or distort the model. |
| 3 | A slicer on `DimCity` should grey out transport modes never used from that city. | **Measure-level** | Do not make the model bidirectional just to change slicer behaviour. Use a measure or visual interaction to test whether a mode has shipments for the selected city; changing the model can create dimension-to-dimension filter propagation and misleading slicers elsewhere. |
| 4 | You want a card showing how many cities appear in `FactShipment` at all. | **Measure-level** | Calculate the distinct city keys in the relevant fact columns inside a measure. Changing relationship direction cannot safely define this role-based count and may change unrelated visual totals. |
| 5 | Row-level security restricts users to their own province. | **Single** | Apply RLS on the province/city dimension and let the filter flow to `FactShipment`. `Both` could propagate fact-derived filters back into dimensions and expose or remove dimension members unexpectedly. |
| 6 | A second fact table, `FactFuelSpend`, shares `DimCity` and `DimDate`. | **Single** | Shared dimensions should filter each fact through separate single-direction relationships. `Both` can create fact-to-fact paths through shared dimensions, causing ambiguous filter propagation and incorrect totals. |

In [1]:
import pandas as pd

cross_filter_register = pd.DataFrame([
    [1, "A slicer on DimMode must filter shipment totals on every visual.", "Single", "Dimension-to-fact filtering is sufficient; Both would add unnecessary reverse propagation."],
    [2, "DimCity and DimMode are both set to Both.", "Single", "Bidirectional paths through FactShipment create ambiguity and can make Power BI reject the model."],
    [3, "DimCity should grey out transport modes never used from that city.", "Measure-level", "Use a measure/visual interaction instead of changing the model; Both would cause unwanted dimension-to-dimension filtering."],
    [4, "Count cities that appear in FactShipment at all.", "Measure-level", "Count distinct fact city keys in a measure; relationship direction cannot safely express both city roles."],
    [5, "RLS restricts users to their own province.", "Single", "The dimension security filter should flow to the fact; Both can propagate fact-derived filters back to dimensions."],
    [6, "FactFuelSpend shares DimCity and DimDate.", "Single", "Shared dimensions should filter each fact independently; Both can create ambiguous fact-to-fact paths."],
], columns=["#", "Situation", "Verdict", "Reason"])
cross_filter_register

,#,Situation,Verdict,Reason
0,1,A slicer on DimMode must filter shipment total...,Single,Dimension-to-fact filtering is sufficient; Bot...
1,2,DimCity and DimMode are both set to Both.,Single,Bidirectional paths through FactShipment creat...
2,3,DimCity should grey out transport modes never ...,Measure-level,Use a measure/visual interaction instead of ch...
3,4,Count cities that appear in FactShipment at all.,Measure-level,Count distinct fact city keys in a measure; re...
4,5,RLS restricts users to their own province.,Single,The dimension security filter should flow to t...
5,6,FactFuelSpend shares DimCity and DimDate.,Single,Shared dimensions should filter each fact inde...


In [2]:
assert len(cross_filter_register) == 6
assert (cross_filter_register["Verdict"] == "Single").sum() == 4
assert (cross_filter_register["Verdict"] == "Measure-level").sum() == 2
assert (cross_filter_register["Verdict"] == "Both").sum() == 0
print("T5 self-checks pass: 4 Single, 2 Measure-level, 0 Both.")

T5 self-checks pass: 4 Single, 2 Measure-level, 0 Both.


## Conclusion

Only the two requirements that need a fact-dependent presentation or calculation are solved at measure level. None of the six requires `Both` in the model.